In [16]:
from langchain_ollama import ChatOllama , OllamaEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage , AIMessage
from langchain_core.prompts import ChatPromptTemplate   
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough

### Load model using Ollama


In [6]:
llm = ChatOllama(model="llama3.2:3b" , temperature=0.7)

question = "What is the capital of France?"

print(llm.invoke([SystemMessage(content="You are a helpful assistant") , HumanMessage(content=question)]))


content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-04-15T16:01:13.2211792Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8801147800, 'load_duration': 5034858100, 'prompt_eval_count': 37, 'prompt_eval_duration': 2465595400, 'eval_count': 8, 'eval_duration': 1258683000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'} id='lc_run--019d91df-eba2-7240-9bfa-b06f0d03f629-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 37, 'output_tokens': 8, 'total_tokens': 45}


### Load the Document

In [7]:
file_path = "UNIT I.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

print(f"Total number of pages: {len(pages)}")

Total number of pages: 24


### Split Document into Chunks


In [23]:
Chunk = RecursiveCharacterTextSplitter(chunk_size=500 , chunk_overlap=100, separators=["\n\n" , "\n" , " "])

chunks = Chunk.split_documents(pages)

print(f"Total number of chunks: {len(chunks)}")

Total number of chunks: 83


### Convert Chunks into Embedding and Store into Chroma Vectore database

In [24]:
embedding = OllamaEmbeddings(model="nomic-embed-text")

vectordb = Chroma.from_documents(chunks , embedding)



### Build a Semantic Retriever , Top-k = 3 (measures the cosine of the angle between two high-dimensional vectors (text embeddings) )

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print(f"chroma index built with {len(vectordb)} vectors.")

results = retriever.invoke("what is tokenization?")
print(results)

print(f"Retrieved {len(results)} chunks for: '{query}'\n")
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:200])
    print()

chroma index built with 126 vectors.
[Document(metadata={'creator': 'Microsoft® Word 2024', 'moddate': '2026-02-05T15:40:35+05:30', 'producer': 'Microsoft® Word 2024', 'total_pages': 24, 'page': 3, 'author': 'Un-named', 'page_label': '4', 'creationdate': '2026-02-05T15:40:35+05:30', 'source': 'UNIT I.pdf'}, page_content='Using NLTK in Python: \n# Tokenizing \nimport nltk \nfrom nltk.tokenize import word_tokenize \nnltk.download(\'punkt_tab\') \ntext = "It focuses on enabling computers to process, understand, and \ngenerate human language. The phrase “Computing with Language” refers \nto the computational treatment of linguistic data—how computers can \nbe programmed to handle text and speech in natural languages. In the \nmodern era, language data is abundant in the form of emails, web'), Document(metadata={'creationdate': '2026-02-05T15:40:35+05:30', 'source': 'UNIT I.pdf', 'producer': 'Microsoft® Word 2024', 'moddate': '2026-02-05T15:40:35+05:30', 'author': 'Un-named', 'total_pages':

### Build a RAG Chain


In [26]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that provides information based on retrieved documents.the provided context. If the context doesn't contain the answer, say 'I don't have enough information to answer this"),
        ("human","Context: {context}\n\nQuestion: {question}"),
    ]
)

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag = ({ "context" : retriever | format_docs ,
       "question" : RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser() )

### Test with Multiple Questions

In [38]:
question = "what is tokenization?" 
res = retriever.invoke(question)
answer = rag.invoke(question)

print("=" * 60)
print(f"Question: {question}")
print("=" * 60)
for i, doc in enumerate(res):
    print(f" Reterive chunk :  {i+1}:\n{doc.page_content}\n{'-'*100}")
print(f"\nAnswer:\n{answer}")

Question: what is tokenization?
 Reterive chunk :  1:
Using NLTK in Python: 
# Tokenizing 
import nltk 
from nltk.tokenize import word_tokenize 
nltk.download('punkt_tab') 
text = "It focuses on enabling computers to process, understand, and 
generate human language. The phrase “Computing with Language” refers 
to the computational treatment of linguistic data—how computers can 
be programmed to handle text and speech in natural languages. In the 
modern era, language data is abundant in the form of emails, web
----------------------------------------------------------------------------------------------------
 Reterive chunk :  2:
specific words like machine, translation, or emotion may appear less often. 
Using NLTK in Python: 
# Tokenizing 
import nltk 
from nltk.tokenize import word_tokenize 
nltk.download('punkt_tab') 
text = "It focuses on enabling computers to process, understand, and 
generate human language. The phrase “Computing with Language” refers 
to the computational tre

In [39]:
question = "what is Egg?" 
res = retriever.invoke(question)
answer = rag.invoke(question)

print("=" * 60)
print(f"Question: {question}")
print("=" * 60)
for i, doc in enumerate(res):
    print(f" Reterive chunk :  {i+1}:\n{doc.page_content}\n{'-'*100}")
print(f"\nAnswer:\n{answer}")

Question: what is Egg?
 Reterive chunk :  1:
• Reasoning & Execution (Reasoning): System decides what action to take 
Generation Pipeline 
• Utterance Planning (Semantics → Syntax) – Decides what to say
----------------------------------------------------------------------------------------------------
 Reterive chunk :  2:
Natural Language Processing (NLP) 
24BAM6C10 
 
UNIT I 
 
 
1. Introduction: Computing with Language 
Natural Language Processing (NLP) is  an AI field combining computer science, 
linguistics, and machine learning to help computers understand, interpret, and generate human 
language (text and speech). It bridges the gap between human communication and machine 
 
 It focuses on enabling computers to process, understand, and generate human language.
----------------------------------------------------------------------------------------------------
 Reterive chunk :  3:
4. Automatic Natural Language Understanding 
Natural Language Understanding (NLU)  is a crucial su